# Markowitz Max-Sharpe Portfolio

**Objective key:** `markowitz` &nbsp;·&nbsp; **Family:** Classical &nbsp;·&nbsp; **Speed:** Fast

## Algorithm

Maximises the Sharpe ratio by solving the mean–variance program as a QP via SLSQP.  
Multi-start restarts guard against local minima from non-convex weight bounds.

The primal form minimised internally is the **negative Sharpe** subject to:
- weights sum to 1
- `weight_min ≤ wᵢ ≤ weight_max` for each asset

## Papers

- **Foundational:** Markowitz (1952) — *Portfolio Selection* — Journal of Finance  
  https://doi.org/10.1111/j.1540-6261.1952.tb01525.x
- **Modern:** Boyd & Vandenberghe (2004) — *Convex Optimization* §4.4 (efficient frontier as QP, free online)  
  https://web.stanford.edu/~boyd/cvxbook/

## Assumptions

- `mu` and `Sigma` are annualised.
- Risk-free rate is assumed zero (Sharpe = return / vol).
- `weight_min=0.005`, `weight_max=0.30`, `n_restarts=8`, `seed=42`.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 10
mu = rng.uniform(0.06, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]

print("Assets:", asset_names)
print("mu:", mu.round(4))

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="markowitz",
    asset_names=asset_names,
    weight_min=0.005,
    weight_max=0.30,
    n_restarts=8,
    seed=42,
)

print(f"\nObjective:       {result.objective}")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nActive weights (w > 0.01):")
for name, w in zip(asset_names, result.weights):
    if w > 0.01:
        print(f"  {name}: {w:.4f}")

In [ ]:
# Compare to Equal Weight baseline
baseline = run_optimization(
    returns=mu, covariance=Sigma, objective="equal_weight", asset_names=asset_names
)
print("Sharpe comparison:")
print(f"  Markowitz: {result.sharpe_ratio:.4f}")
print(f"  EqualWgt:  {baseline.sharpe_ratio:.4f}")
print(f"  Lift:      {result.sharpe_ratio - baseline.sharpe_ratio:+.4f}")